In [ ]:
import sys
import json
import re
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

def parse_line(line):
    # Regex to extract UNIX time (e.g., 1764000892.487560)
    unix_time_match = re.search(r'(\d+\.\d+)', line)
    if not unix_time_match:
        return None
    unix_time = float(unix_time_match.group(1))

    # Regex to extract the entire JSON object starting with "class":"TPV"
    json_match = re.search(r'\{.*?"class":"TPV".*?\}', line, re.DOTALL)
    if not json_match:
        return None
    json_str = json_match.group(0)

    try:
        data = json.loads(json_str)
    except json.JSONDecodeError:
        return None

    # Extract the required fields
    lat = data.get('lat')
    lon = data.get('lon')
    alt = data.get('alt')
    altMSL = data.get('altMSL')
    ecefx = data.get('ecefx')
    ecefy = data.get('ecefy')
    ecefz = data.get('ecefz')

    return {
        'unix_time': unix_time,
        'lat': lat,
        'lon': lon,
        'alt': alt,
        'altMSL': altMSL,
        'ecefx': ecefx,
        'ecefy': ecefy,
        'ecefz': ecefz,
    }

def extract_locations_from_gpspipe(filename):
    results = []

    try:
        with open(filename, 'r') as f:
            for line in f:
                if 'TPV' in line:
                    result = parse_line(line)
                    if result:
                        results.append(result)
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return
    
    # Convert list of dicts to DataFrame
    df = pd.DataFrame(results)

    # Uncomment below to save as CSV
    # df.to_csv('output.csv', index=False)
    return df
    

def main():
    filename = '/Users/thatch/Downloads/20251124_161604_gpspipe_stdout.log'  # Replace with your file path if needed
    filename = '/tmp/SORA-1202/20251202_183128_gpspipe_stdout.log'
    
    df = extract_locations_from_gpspipe(filename)

    # Optional: Print the DataFrame or save it as CSV
    print(df)
    # Uncomment below to save as CSV
    # df.to_csv('output.csv', index=False)
    return df

if __name__ == '__main__':
    gpsdf = main()


In [ ]:
gpsdf = main()

In [ ]:
def get_distance_along_track(df):
    displacements = np.linalg.norm(df.iloc[1:,:].loc[:, ["ecefx", "ecefy", "ecefz"]].to_numpy() - \
    df.iloc[:-1,:].loc[:, ["ecefx", "ecefy", "ecefz"]].to_numpy(), axis=1)
    return np.cumsum(displacements)


In [ ]:
np.shape(get_distance_along_track(gpsdf))

In [ ]:
gpsdf['unix_time']

In [ ]:
np.shape(gpsdf['unix_time'][1:].to_numpy())

In [ ]:
interp_dist = interp1d(gpsdf['unix_time'][1:].to_numpy(), get_distance_along_track(gpsdf), kind='linear')


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, axs = plt.subplots(facecolor='white', figsize=(10,6),nrows=2)

axs[0].plot(gpsdf["unix_time"], gpsdf['lat'], label='Latitude')
axs[0].set_xlabel('Time (UNIX)')
axs[0].set_ylabel('Lat (deg)')
axs[0].set_title('Latitude')
axs[0].grid()
axs[1].plot(gpsdf["unix_time"], gpsdf['lat'], label='Longitude')
axs[1].set_xlabel('Time (UNIX)')
axs[1].set_ylabel('Lng (deg)')
axs[1].set_title('Latitude')
axs[1].grid()
#axs[1].legend()

In [ ]:
from scipy.interpolate import interp1d

In [ ]:
# Interpolate latitude and longitude as functions of time
f_lat = interp1d(gpsdf['unix_time'], gpsdf['lat'], kind='linear')
f_lon = interp1d(gpsdf['unix_time'], gpsdf['lon'], kind='linear')